In [1]:
import mpramnist
from mpramnist.Agarwal2025.dataset import AgarwalSingleDataset
from mpramnist.Kircher2019.dataset import KircherDataset

from mpramnist.Kircher2019.trainer import LitModel_Kircher

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights


import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1024
NUM_WORKERS = 8

# Kircher Dataset and Experimental Design

The Kircher dataset is based on **MPRA (Massively Parallel Reporter Assay)** results from **saturation mutagenesis** experiments. The study characterized **44,647** variants of regulatory elements, including promorers and enhancers across multiple cell lines:

### Promoters

| Name | Element Type | Cell type |
| ----------- | ----------- | ----------- |
| F9 | Hemophilia B | HepG2 |
| LDLR | Familial hypercholesterolemia | HepG2 |
| FOXE1 | Thyroid cancer | HeLa |
| GP1BB | Bernard-Soulier Syndrome |  HEL 92.1.7 |
| HBB | Thalassemia | HEL 92.1.7 |
| HBG1 | Hereditary persistence of fetal hemoglobin | HEL 92.1.7 |
| HNF4A (P2) | Maturity-onset diabetes of the young (MODY) | HEK293T |
| MSMB | Prostate cancer | HEK293T |
| TERT | Various types of cancer | HEK293T, SF7996 |
| PKLR | Pyruvate kinase deficiency | K562 |


### Enhancers 

| Name | Element Type | Cell type |
| ----------- | :----------- | ----------- |
| SORT1 | Plasma low-density lipoprotein cholesterol & myocardial infarction | HepG2 |
| BCL11A+58 | Sickle cell disease | HEL 92.1.7 |
| MYC (rs6983267) | Various types of cancer | HEK293T | 
| IRF6 | Cleft lip | HaCaT |
| IRF4 | Human pigmentation | SK-MEL-28 |
| MYC (rs11986220) | Various types of cancer | LNCaP + 100nM DHT |
| RET | Hirschsprung | Neuro-2a |
| UC88 | - | Neuro-2a |
| TCF7L2 | Type 2 diabetes | MIN6 |
| ZFAND3 | Type 2 diabetes | MIN6 | 
| ZRS | Limb malformations | NIH-3T3 (with HOXD13/ HOXD13+HAND2) |


# Proposed KircherDataset Benchmark Application

These experimentally validated sequences are recommended as a reference dataset for evaluating the performance of machine learning models.

# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **AgarwalSingle dataset**

2. Assess its predictive power using **Kircher's saturation mutagenesis data**

Focus: Promoters and enhancers tested in **HepG2** cell line

# Pretrain on Agarwal's HepG2

In [2]:
# shift (0,15)
# preprocessing
shift = 15

train_transform = t.Compose(
    [
        t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK),
        t.AddFlanks("", AgarwalSingleDataset.RIGHT_FLANK),
        t.RightCrop(230, 260),  
        t.LeftCrop(230, 230),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
test_transform = t.Compose(
    [
        t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

# load the data
Cell_Type = "HepG2" # or K562 
train_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="train", transform=train_transform, root="../data/",)
val_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="val", transform=test_transform, root="../data/",)
test_dataset = AgarwalSingleDataset(cell_type=Cell_Type, split="test", transform=test_transform, root="../data/",)

# encapsulate data into dataloader form
train_loader = DataLoader(dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=103)
val_loader = DataLoader(dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=103)
test_loader = DataLoader(dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=103)

print(len(train_dataset), len(val_dataset), len(test_dataset))

in_channels = len(train_dataset[0][0])
out_channels = 1

98336 12292 12298


In [3]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Kircher(model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10)

checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)
# Initialize a trainer
trainer_hepg2 = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=5,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    callbacks=[checkpoint_callback],
)

trainer_hepg2.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.

Epoch 4: 100%|██████████| 97/97 [00:16<00:00,  5.93it/s, v_num=0, val_loss=0.307, val_pearson=0.701, train_loss=0.270]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 97/97 [00:16<00:00,  5.89it/s, v_num=0, val_loss=0.307, val_pearson=0.701, train_loss=0.270]


In [4]:
best_model_path = checkpoint_callback.best_model_path
seq_model_hepg2 = LitModel_Kircher.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1,)

## Kircher's sequences evaluation

In [5]:
elements_for_hepg2 = ["F9", "LDLR.2", "LDLR", "SORT1.2", "SORT1", "SORT1-flip"]

In [6]:
def meaned_prediction(forw, rev, trainer, seq_model, name, is_kircher=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print(name, " Pearson correlation")

    if is_kircher:
        y_preds_forw_alt = torch.cat(
            [pred["alt_predicted"] for pred in predictions_forw]
        )
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets)

    return pears(mean_forw, targets)

In [8]:
forw_transform = t.Compose([t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK), t.Seq2Tensor()])
rev_transform = t.Compose([t.AddFlanks(AgarwalSingleDataset.CONSTANT_LEFT_FLANK, AgarwalSingleDataset.CONSTANT_RIGHT_FLANK),t.ReverseComplement(1),t.Seq2Tensor(),])

ldlr_dataset_forw = KircherDataset(length=200, elements=["LDLR", "LDLR.2"], transform=forw_transform, root="../data/")
ldlr_dataset_rev = KircherDataset(length=200, elements=["LDLR", "LDLR.2"], transform=rev_transform, root="../data/")

print("LDLR info")
print(ldlr_dataset_forw)

ldlr_forw = DataLoader(dataset=ldlr_dataset_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
ldlr_rev = DataLoader(dataset=ldlr_dataset_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

print("==========")
print(meaned_prediction(ldlr_forw,ldlr_rev,trainer_hepg2,seq_model_hepg2,name="LDLR",is_kircher=True,))
print("==========")

f9_dataset_forw = KircherDataset(length=200, elements=["F9"], transform=forw_transform, root="../data/")
f9_dataset_rev = KircherDataset(length=200, elements=["F9"], transform=rev_transform, root="../data/")

print("F9 info")
print(f9_dataset_forw)

f9_forw = DataLoader(dataset=f9_dataset_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
f9_rev = DataLoader(dataset=f9_dataset_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

print("==========")
print(meaned_prediction(f9_forw,f9_rev,trainer_hepg2,seq_model_hepg2,name="F9",is_kircher=True,))
print("==========")

sort1_dataset_forw = KircherDataset(length=200, elements=["SORT1.2", "SORT1", "SORT1-flip"], transform=forw_transform, root="../data/")
sort1_dataset_rev = KircherDataset(length=200, elements=["SORT1.2", "SORT1", "SORT1-flip"], transform=rev_transform, root="../data/")

print("SORT1 info")
print(sort1_dataset_forw)

sort1_forw = DataLoader(dataset=sort1_dataset_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
sort1_rev = DataLoader(dataset=sort1_dataset_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
print("==========")
print(meaned_prediction(sort1_forw,sort1_rev,trainer_hepg2,seq_model_hepg2,name="F9",is_kircher=True,))
print("==========")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


LDLR info
Dataset KircherDataset (MpraDaraset)
    Number of datapoints: 2176
    Root location: ../data/Kircher
    Using split: test
    Split: {'HepG2': ['F9', 'LDLR', 'LDLR.2', 'SORT1', 'SORT1-flip', 'SORT1.2'], 'HeLa': ['FOXE1'], 'HEL92.1.7': ['GP1BA', 'HBB', 'HBG1', 'BCL11A'], 'HEK293T': ['HNF4A', 'MSMB', 'TERT-HEK', 'MYCrs6983267'], 'K562': ['PKLR-24h', 'PKLR-48h'], 'SF7996': ['TERT-GAa', 'TERT-GBM', 'TERT-GSc'], 'SK-MEL-28': ['IRF4'], 'HaCaT': ['IRF6'], 'LNCaP': ['MYCrs11986220'], 'Neuro-2a': ['RET', 'UC88'], 'MIN6': ['TCF7L2', 'ZFAND3'], 'NIH-3T3': ['ZRSh-13', 'ZRSh-13h2']}
    Task: Regression
    Description: The Kircher dataset is based on MPRA saturation mutagenesis results and includes 44,647 variants of regulatory elements tested in various cell lines. These data are proposed to be used as a benchmark set for evaluating the quality of machine learning models. The regression task involved predicting a scalar value of regulatory activity for the corresponding cell line.
Pr

Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 41.85it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]



Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 40.54it/s]
LDLR  Pearson correlation
tensor(0.5186)


...

Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 29.32it/s]
F9  Pearson correlation
tensor(0.4912)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


SORT1 info
Dataset KircherDataset (MpraDaraset)
    Number of datapoints: 5898
    Root location: ../data/Kircher
    Using split: test
    Split: {'HepG2': ['F9', 'LDLR', 'LDLR.2', 'SORT1', 'SORT1-flip', 'SORT1.2'], 'HeLa': ['FOXE1'], 'HEL92.1.7': ['GP1BA', 'HBB', 'HBG1', 'BCL11A'], 'HEK293T': ['HNF4A', 'MSMB', 'TERT-HEK', 'MYCrs6983267'], 'K562': ['PKLR-24h', 'PKLR-48h'], 'SF7996': ['TERT-GAa', 'TERT-GBM', 'TERT-GSc'], 'SK-MEL-28': ['IRF4'], 'HaCaT': ['IRF6'], 'LNCaP': ['MYCrs11986220'], 'Neuro-2a': ['RET', 'UC88'], 'MIN6': ['TCF7L2', 'ZFAND3'], 'NIH-3T3': ['ZRSh-13', 'ZRSh-13h2']}
    Task: Regression
    Description: The Kircher dataset is based on MPRA saturation mutagenesis results and includes 44,647 variants of regulatory elements tested in various cell lines. These data are proposed to be used as a benchmark set for evaluating the quality of machine learning models. The regression task involved predicting a scalar value of regulatory activity for the corresponding cell line.
P

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 35.70it/s]
F9  Pearson correlation
tensor(0.3120)


In [ ]:
elements_for_k562 = ["PKLR-24h", "PKLR-48h"]